# ?? ???? ?? ????


### ??? ????


In [ ]:
import os
from pprint import pprint

import optuna
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
import pandas as pd

THRESHOLD = 0.3
RANDOM_STATE = 110

train_data = pd.read_csv("../../../data/train_data_0817.csv")
test_data = pd.read_csv("../../../data/test_data_0817.csv")

In [ ]:
# dam, fill1, fill2 ?? ??
var_dam_fill = [
    'Equipment_same_num',
    'PalletID_Collect_Result_encoded',
    'Production_Qty_Collect_Result',
    'WorkMode Collect Result'
]

In [ ]:
# ?? ?? ??
### correlation ??? ?? ?? ???
var_all_corr = [
    'model_receip_encoded',
    'workorder_receip_encoded'
]

### train
var_all_train = [
    'target',
    'model_receip_encoded',
    'workorder_receip_encoded'
]

### test
var_all_test = [
    'Set ID',
    'target',
    'model_receip_encoded',
    'workorder_receip_encoded'
]

In [ ]:
# '_Dam'? ???? ?? ??
dam_variables = [var for var in train_data.columns if '_Dam' in var]

# train
final_columns_train = var_dam_fill + var_all_train + dam_variables
train_data_dam = train_data[final_columns_train]

# test 
final_columns_test = var_dam_fill + var_all_test + dam_variables
test_data_dam = test_data[final_columns_test]

In [ ]:
# '_Fill1'? ???? ?? ??
fill1_variables = [var for var in train_data.columns if '_Fill1' in var]

# train
final_columns_train = var_dam_fill + var_all_train + fill1_variables
train_data_fill1 = train_data[final_columns_train]

# test 
final_columns_test = var_dam_fill + var_all_test + fill1_variables
test_data_fill1 = test_data[final_columns_test]

In [ ]:
# '_Fill2'? ???? ?? ??
fill2_variables = [var for var in train_data.columns if '_Fill2' in var]

# train
final_columns_train = var_dam_fill + var_all_train + fill2_variables
train_data_fill2 = train_data[final_columns_train]

# test 
final_columns_test = var_dam_fill + var_all_test + fill2_variables
test_data_fill2 = test_data[final_columns_test]

In [ ]:
# '_AutoClave'? ???? ?? ??
autoclave_variables = [var for var in train_data.columns if '_AutoClave' in var]

# train
final_columns_train = var_all_train + autoclave_variables
train_data_autoclave = train_data[final_columns_train]

# test 
final_columns_test = var_all_test + autoclave_variables
test_data_autoclave = test_data[final_columns_test]

In [ ]:
# ? DataFrame? ?? ? ??
num_columns_train_data = train_data.shape[1]
num_columns_train_data_dam = train_data_dam.shape[1]
num_columns_train_data_autoclave = train_data_autoclave.shape[1]
num_columns_train_data_fill1 = train_data_fill1.shape[1]
num_columns_train_data_fill2 = train_data_fill2.shape[1]

num_columns_test_data = test_data.shape[1]
num_columns_test_data_dam = test_data_dam.shape[1]
num_columns_test_data_autoclave = test_data_autoclave.shape[1]
num_columns_test_data_fill1 = test_data_fill1.shape[1]
num_columns_test_data_fill2 = test_data_fill2.shape[1]

# ? DataFrame? ?? ? ??
print("----train data-----")
print(f"train_data DataFrame? ?? ?: {num_columns_train_data}")
print(f"train_data_dam DataFrame? ?? ?: {num_columns_train_data_dam}")
print(f"train_data_autoclave DataFrame? ?? ?: {num_columns_train_data_autoclave}")
print(f"train_data_fill1 DataFrame? ?? ?: {num_columns_train_data_fill1}")
print(f"train_data_fill2 DataFrame? ?? ?: {num_columns_train_data_fill2}")
print("----test data-----")
print(f"test_data DataFrame? ?? ?: {num_columns_test_data}")
print(f"test_data_dam DataFrame? ?? ?: {num_columns_test_data_dam}")
print(f"test_data_autoclave DataFrame? ?? ?: {num_columns_test_data_autoclave}")
print(f"test_data_fill1 DataFrame? ?? ?: {num_columns_test_data_fill1}")
print(f"test_data_fill2 DataFrame? ?? ?: {num_columns_test_data_fill2}")

---

## Optuna

????? 0.3?? ?????? ?? ??? ?

Trial 128 finished with value: 0.1932938856015779  
parameters: {'n_estimators': 1298, 'max_depth': 20, 'min_samples_split': 6, 'min_samples_leaf': 1,  
'criterion': 'entropy', 'bootstrap': False}

In [ ]:
# # 'Normal'? 'AbNormal'? ??? ??
# train_data_fill1['target'] = train_data_fill1['target'].map({'Normal': 0, 'AbNormal': 1})

# # ????? ??
# THRESHOLD = 0.3

# def objectiveRandomForestClassifier(trial, x_tr, y_tr, x_val, y_val):
#     param = {
#         'n_estimators' : trial.suggest_int('n_estimators', 800, 3000),
#         'max_depth' : trial.suggest_int('max_depth', 20, 100),
#         'min_samples_split' : trial.suggest_int('min_samples_split', 2, 20),
#         'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 8),
#         'criterion' : trial.suggest_categorical("criterion", ["entropy",]),
#         'class_weight' : trial.suggest_categorical("class_weight", ["balanced"]),
#         'random_state': RANDOM_STATE
#     }
    
#     model = RandomForestClassifier(**param)
#     model.fit(x_tr, y_tr)
#     pred = model.predict(x_val)
#     score = f1_score(y_val, pred, average="binary")
    
#     return score

# # ???? ??
# x_train, x_val, y_train, y_val = train_test_split(
#     train_data_fill1.drop("target", axis=1),
#     train_data_fill1["target"],
#     test_size=0.2,
#     shuffle=True,
#     random_state=RANDOM_STATE,
# )

# # ??? ???? ??
# study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
# study.optimize(lambda trial: objectiveRandomForestClassifier(trial, x_train, y_train, x_val, y_val), n_trials=300)

# print('Best trial: score {}, \nparams {}'.format(study.best_trial.value, study.best_trial.params))

Best trial: score 0.23151125401929262,  
params {'n_estimators': 804, 'max_depth': 69, 'min_samples_split': 7, 'min_samples_leaf': 5, 'criterion': 'entropy', 'class_weight': 'balanced'}  

In [ ]:
# 'Normal'? 'AbNormal'? ??? ??
train_data_fill1['target'] = train_data_fill1['target'].map({'Normal': 0, 'AbNormal': 1})

# ????? ??
THRESHOLD = 0.3

def objectiveRandomForestClassifier(trial, x_tr, y_tr, x_val, y_val):
    param = {
        'n_estimators' : trial.suggest_int('n_estimators', 800, 3000),
        'max_depth' : trial.suggest_int('max_depth', 20, 100),
        'min_samples_split' : trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 8),
        'criterion' : trial.suggest_categorical("criterion", ["entropy",]),
        'class_weight' : trial.suggest_categorical("class_weight", ["balanced"]),
        'random_state': RANDOM_STATE
    }
    
    model = RandomForestClassifier(**param)
    model.fit(x_tr, y_tr)
    pred_proba = model.predict_proba(x_val)[:, 1]  # ?? ??? ??
    pred = (pred_proba >= THRESHOLD).astype(int)  # ?????? ?? ??
    
    score = f1_score(y_val, pred, average="binary")
    
    return score

# ???? ??
x_train, x_val, y_train, y_val = train_test_split(
    train_data_fill1.drop("target", axis=1),
    train_data_fill1["target"],
    test_size=0.2,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# ??? ???? ??
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(lambda trial: objectiveRandomForestClassifier(trial, x_train, y_train, x_val, y_val), n_trials=300)

print('Best trial: score {}, \nparams {}'.format(study.best_trial.value, study.best_trial.params))

Best trial: score 0.2207670720299345,  
params {'n_estimators': 1861, 'max_depth': 91, 'min_samples_split': 5, 'min_samples_leaf': 1, 'criterion': 'entropy', 'class_weight': 'balanced'}

.